Upload aoi.py to Colab

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving aoi.py to aoi.py


Install required packages & import the module

In [ ]:
!pip install -q geopandas pyogrio shapely

import sys
import os
from pathlib import Path

# Add current working directory to sys.path so Python finds aoi.py
sys.path.append(os.path.abspath("."))

os.makedirs("src", exist_ok=True)
with open("src/config.py", "w") as f:
    f.write("from pathlib import Path\nREPO_ROOT = Path('.')\n")

from aoi import build_aoi

Adding Kilometrage data

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving kilometrage-des-cours-deau.gpkg to kilometrage-des-cours-deau.gpkg


Run build_aoi() with project parameters

In [ ]:
# Define your inputs
RIVER_NAME = "Alzette"
KM_START = 35
KM_END = 50
MARGIN_METERS = 500.0
DATA_PATH = "kilometrage-des-cours-deau.gpkg"  # Path to your uploaded river kilometrage file

# Execute your teammate's function
aoi_gdf, points_gdf = build_aoi(
    river=RIVER_NAME,
    km_start=KM_START,
    km_end=KM_END,
    margin_m=MARGIN_METERS,
    kilometrage_path=DATA_PATH
)

# Inspect the created Area of Interest
print("AOI successfully generated:")
print(aoi_gdf)

# Quick visualization check
#aoi_gdf.plot()

AOI successfully generated:
     river  km_start  km_end  margin_m method  n_points  \
0  Alzette        35      50     500.0   bbox       151   

                                            geometry  
0  POLYGON ((80682.526 68051.79, 80682.526 76766....  


# Seperate the Buildings of AOI

In [ ]:
import geopandas as gpd

# Generate AOI polygon
RIVER_NAME = "Alzette"
KM_START = 35
KM_END = 50
MARGIN_METERS = 500.0
DATA_PATH = "kilometrage-des-cours-deau.gpkg"

aoi_gdf, points_gdf = build_aoi(
    river=RIVER_NAME,
    km_start=KM_START,
    km_end=KM_END,
    margin_m=MARGIN_METERS,
    kilometrage_path=DATA_PATH
)

print(f"AOI CRS: {aoi_gdf.crs}")

# Read GML buildings WITHOUT initial bbox filtering first (to check native CRS)
buildings = gpd.read_file("BU.Building.gml")
print(f"Raw GML Buildings Count: {len(buildings)}")
print(f"GML Buildings CRS: {buildings.crs}")

# Match CRS BEFORE spatial filtering
if buildings.crs != aoi_gdf.crs:
    buildings = buildings.to_crs(aoi_gdf.crs)

# Filter buildings intersecting the AOI polygon
aoi_polygon = aoi_gdf.unary_union
filtered_buildings = buildings[buildings.intersects(aoi_polygon)].copy()

print(f"Filtered Buildings Count: {len(filtered_buildings)}")

# Convert 3D geometries to 2D safely
filtered_buildings["geometry"] = filtered_buildings.geometry.force_2d()

# Save back to GeoPackage
filtered_buildings.to_file("alzette_aoi_buildings.gpkg", driver="GPKG")
print("Saved non-empty buildings GPKG!")

AOI CRS: EPSG:2169


/usr/local/lib/python3.13/dist-packages/pyogrio/raw.py:200: RuntimeWarning: Field with same name (description) already exists in (Building). Skipping newer ones
  return ogr_read(
/usr/local/lib/python3.13/dist-packages/pyogrio/raw.py:200: RuntimeWarning: Field with same name (identifier) already exists in (Building). Skipping newer ones
  return ogr_read(


Raw GML Buildings Count: 240495
GML Buildings CRS: EPSG:3035


/tmp/ipykernel_2168/3540071574.py:30: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  aoi_polygon = aoi_gdf.unary_union


Filtered Buildings Count: 14683
Saved non-empty buildings GPKG!


## Commune & Population + Height & Volume

* Adding EUBUCCO data
* Apply a size-based dynamic fallback height -> only for NAN data.

In [ ]:
import geopandas as gpd

# Read the vector file (replace with your file name)
file_path = "LU00.gpkg"  # or "your_file.shp"
gdf = gpd.read_file(file_path)

# Check the Coordinate Reference System (CRS)
print("--- Coordinate System ---")
print(gdf.crs)

# View column names (look for: 'height', 'z', 'elevation', 'contour', 'HGT', 'ALT')
print("\n--- Available Columns ---")
print(gdf.columns.tolist())

# Preview the first 5 rows of data
print("\n--- Data Preview ---")
gdf.head()

--- Coordinate System ---
EPSG:3035

--- Available Columns ---
['id', 'region_id', 'city_id', 'type', 'subtype', 'height', 'floors', 'construction_year', 'type_confidence', 'subtype_confidence', 'height_confidence_lower', 'height_confidence_upper', 'floors_confidence_lower', 'floors_confidence_upper', 'construction_year_confidence_lower', 'construction_year_confidence_upper', 'geometry_source', 'type_source', 'subtype_source', 'height_source', 'floors_source', 'construction_year_source', 'geometry_source_id', 'type_source_ids', 'subtype_source_ids', 'height_source_ids', 'floors_source_ids', 'construction_year_source_ids', 'subtype_raw', 'geometry']

--- Data Preview ---


,id,region_id,city_id,type,subtype,height,floors,construction_year,type_confidence,subtype_confidence,...,floors_source,construction_year_source,geometry_source_id,type_source_ids,subtype_source_ids,height_source_ids,floors_source_ids,construction_year_source_ids,subtype_raw,geometry
0,0010e82170824bfe-0,LU000,LU0101,residential,detached,8.1,1.4,NaN,0.82,0.76,...,estimated,nan,luxembourg-latest_168784,None,None,None,None,None,None,"POLYGON ((4030192.443 2949215.064, 4030201.137..."
1,01b93ac4f3ee4040-0,LU000,LU0101,non-residential,industrial,9.7,1.8,NaN,0.92,0.44,...,estimated,nan,luxembourg-latest_85779,None,None,None,None,None,None,"POLYGON ((4029962.495 2949250.53, 4029975.534 ..."
2,025eaef0a7a042f7-0,LU000,LU0101,residential,detached,8.2,1.7,NaN,NaN,0.81,...,estimated,nan,luxembourg-latest_128092,"[""luxembourg-latest_128092""]",None,None,None,None,house,"POLYGON ((4028751.793 2948784.317, 4028755.552..."
3,025eaef0a7a042f7-1,LU000,LU0101,residential,semi-detached,7.5,1.6,NaN,NaN,0.34,...,estimated,nan,luxembourg-latest_128093,"[""luxembourg-latest_128093""]",None,None,None,None,house,"POLYGON ((4028763.095 2948777.98, 4028767.704 ..."
4,030608a204854ee9-0,LU000,LU0101,residential,apartment,9.8,3.0,NaN,NaN,NaN,...,osm,nan,luxembourg-latest_82887,"[""luxembourg-latest_82887""]","[""luxembourg-latest_82887""]",None,"[""luxembourg-latest_82887""]",None,apartments,"POLYGON ((4029150.834 2949292.037, 4029159.28 ..."


This code loads both datasets at the beginning, performs the spatial join to extract actual building heights (height and floors), fills in missing heights using your fallback area logic, and calculates volume and dasymetric population weights.

final_height_m is single consolidated height column. It acts as a smart combined variable:
* If real data exists: It takes the measured height from LU00.gpkg (ext_height).
* If real data is missing (NaN or <= 0): It automatically uses the estimated height calculated from your building's footprint area (determine_final_height).

In [ ]:
import os
import numpy as np
import geopandas as gpd
import pandas as pd

# Configuration and Setup
os.environ["SHAPE_RESTORE_SHX"] = "YES"
TARGET_CRS = "EPSG:2169"  # Luxembourg LUREF projection for accurate metric calculations

# Load Datasets
commune_gdf = gpd.read_file("LIMADM_COMMUNES.shp")
population_df = pd.read_csv("rnpp_population_per_municipality.csv")
raw_buildings = gpd.read_file("alzette_aoi_buildings.gpkg")
height_gpkg = gpd.read_file("LU00.gpkg")

# Lowercase column names across datasets
commune_gdf.columns = commune_gdf.columns.str.strip().str.lower()
population_df.columns = population_df.columns.str.strip().str.lower()
raw_buildings.columns = raw_buildings.columns.str.strip().str.lower()
height_gpkg.columns = height_gpkg.columns.str.strip().str.lower()

# Standardize Commune Identifiers
def standardize_commune_id(df, name):
    possible_keys = ["lau2", "la_id", "commune_code", "id_commune", "id"]
    for key in possible_keys:
        if key in df.columns:
            if key != "lau2":
                df = df.rename(columns={key: "lau2"})
            return df
    raise KeyError(f"Could not find a valid commune identifier column in {name}.")

commune_gdf = standardize_commune_id(commune_gdf, "LIMADM_COMMUNES.shp")
population_df = standardize_commune_id(population_df, "rnpp_population_per_municipality.csv")

commune_gdf["lau2"] = commune_gdf["lau2"].astype(str).str.strip().str.zfill(4)
population_df["lau2"] = population_df["lau2"].astype(str).str.strip().str.zfill(4)

# Prepare Building Geometries & Extract Physical Heights
buildings = raw_buildings.explode(index_parts=False).reset_index(drop=True)

# Project to metric CRS and calculate 2D footprint area
buildings = buildings.to_crs(TARGET_CRS)
buildings["geometry"] = buildings.geometry.force_2d()
buildings["footprint_m2"] = buildings.geometry.area

# Spatial Join with LU00.gpkg to Get External Height & Floor Data
# Match using building centroids in LU00's CRS
height_gpkg_clean = height_gpkg.drop(columns=['index_right', 'index_left'], errors='ignore')
buildings_in_gpkg_crs = buildings.to_crs(height_gpkg.crs)
buildings_in_gpkg_crs["geometry"] = buildings_in_gpkg_crs.geometry.centroid

merged_heights = gpd.sjoin_nearest(
    buildings_in_gpkg_crs,
    height_gpkg_clean[['height', 'floors', 'type', 'geometry']],
    how="left",
    max_distance=30,  # 30 meters tolerance
    distance_col="dist_to_building_m"
)

# Remove duplicate matches created by distance ties ---
merged_heights = merged_heights[~merged_heights.index.duplicated(keep='first')]

# Bring joined height and floors attributes back into main buildings GeoDataFrame
buildings["ext_height"] = merged_heights["height"]
buildings["floors"] = merged_heights["floors"]
buildings["building_type"] = merged_heights["type"]

# Fallback Height Logic
def determine_final_height(row):
    # Use real height from LU00.gpkg if available
    if pd.notna(row["ext_height"]) and row["ext_height"] > 0:
        return row["ext_height"]

    # Fallback estimated based on footprint area
    area = row["footprint_m2"]
    if area < 60:
        return 5.0   # Small outbuilding / garage
    elif area < 180:
        return 9.0   # Standard house (~3 floors)
    elif area < 400:
        return 12.0  # Large residential / townhouse
    else:
        return 18.0  # Large apartment block / commercial

buildings["final_height_m"] = buildings.apply(determine_final_height, axis=1)
buildings["volume_m3"] = buildings["footprint_m2"] * buildings["final_height_m"]

# Aggregate Commune Population Data
pop_columns = [col for col in ["femmes_mineures", "hommes_mineurs", "femmes_majeures", "hommes_majeures"] if col in population_df.columns]
if pop_columns:
    population_df["population"] = population_df[pop_columns].sum(axis=1)

communes_with_pop = commune_gdf.merge(population_df, on="lau2", how="left").to_crs(TARGET_CRS)

# Spatial Join Buildings to Communes
centroids = buildings.copy()
centroids["geometry"] = centroids.geometry.centroid

buildings_joined = gpd.sjoin(
    centroids,
    communes_with_pop,
    how="inner",
    predicate="intersects"
)

# Re-assign polygon geometry
buildings_joined["geometry"] = buildings.loc[buildings_joined.index, "geometry"]

# Dasymetric Population Disaggregation by Building Volume
commune_totals = (
    buildings_joined.groupby("lau2")["volume_m3"]
    .sum()
    .reset_index(name="total_commune_volume_m3")
)

buildings_joined = buildings_joined.merge(commune_totals, on="lau2", how="left")
buildings_joined["vol_weight"] = buildings_joined["volume_m3"] / buildings_joined["total_commune_volume_m3"]
buildings_joined["est_population"] = buildings_joined["vol_weight"] * buildings_joined["population"]

# Extract Projected (EPSG:2169) & Lat/Lon (EPSG:4326) Coordinates
buildings_joined["x_coord_2169"] = buildings_joined.geometry.centroid.x
buildings_joined["y_coord_2169"] = buildings_joined.geometry.centroid.y

centroids_wgs84 = buildings_joined.geometry.centroid.to_crs(epsg=4326)
buildings_joined["longitude"] = centroids_wgs84.x
buildings_joined["latitude"] = centroids_wgs84.y

# 11. Export Cleaned Table to CSV
export_df = pd.DataFrame(buildings_joined.drop(columns=["geometry", "index_right", "index_left",
                                                        "ext_height", "canton", "commune_nom",
                                                        "femmes_mineures", "hommes_mineurs",
                                                        "femmes_majeures", "hommes_majeurs",
                                                        "population","tota_commune_volume_m3",
                                                        "vol_weight"], errors="ignore"))
export_df.to_csv("buildings_population_final.csv", index=False)

preview_cols = ["lau2", "commune", "building_type", "latitude", "longitude",
                "footprint_m2", "final_height_m", "floors", "volume_m3",
                "est_population"]

print(export_df[[col for col in preview_cols if col in export_df.columns]].head(10))

   lau2     commune    building_type   latitude  longitude  footprint_m2  \
0  0304  Luxembourg      residential  49.602975   6.145402     75.416233   
1  0304  Luxembourg      residential  49.596263   6.151966     71.357460   
2  0304  Luxembourg      residential  49.589425   6.125509     87.615363   
3  0303  Hesperange      residential  49.568744   6.166306    149.622259   
4  0303  Hesperange      residential  49.582245   6.143665    184.738126   
5  0304  Luxembourg      residential  49.617185   6.158444     71.244404   
6  0304  Luxembourg      residential  49.600208   6.142932     69.435489   
7  0304  Luxembourg  non-residential  49.616786   6.155486     27.400915   
8  0303  Hesperange  non-residential  49.569780   6.148017     26.368378   
9  0304  Luxembourg      residential  49.609672   6.130032    240.694356   

   final_height_m  floors    volume_m3  est_population  
0             8.3     2.5   625.954737        0.560143  
1             6.1     1.8   435.280504        0.3

In [ ]:
# Check how many buildings are missing external height data
missing_heights = buildings["ext_height"].isna().sum()
zero_heights = (buildings["ext_height"] <= 0).sum()

print(f"Buildings using LU00 height: {len(buildings) - missing_heights - zero_heights}")
print(f"Buildings relying on fallback logic: {missing_heights + zero_heights}")

Buildings using LU00 height: 14645
Buildings relying on fallback logic: 39


Getting ready for EDDIE:

What EDDIE will read in your GeoJSON:

All calculated fields will be preserved inside the GeoJSON feature properties:
* latitude & longitude
* building_type
* final_height_m
* floors
* volume_m3
* est_population (Estimated population per building)
* lau2 & commune name

In [ ]:
import geopandas as gpd

# Convert the joined building polygons back to standard WGS84 (lat/lon)
# 'buildings_joined' contains the original 2D building footprint polygons
polygons_wgs84 = buildings_joined.to_crs("EPSG:4326")

# Drop intermediate index columns to keep the file clean
polygons_wgs84 = polygons_wgs84.drop(columns=["index_right", "index_left",
                                            "ext_height", "canton", "commune_nom",
                                            "femmes_mineures", "hommes_mineurs",
                                            "femmes_majeures", "hommes_majeurs",
                                            "population","total_commune_volume_m3",
                                            "vol_weight"], errors="ignore")

# Export as GeoJSON with full polygon geometries
polygons_wgs84.to_file("buildings_polygons_eddie.geojson", driver="GeoJSON")

print("Successfully exported 'buildings_polygons_eddie.geojson' with full building shapes!")

Successfully exported 'buildings_polygons_eddie.geojson' with full building shapes!


make 3d polygons

In [ ]:
import geopandas as gpd
from shapely.ops import transform

# Start with projected metric GeoDataFrame (EPSG:2169)
# (Working in meters makes Z-height additions accurate)
buildings_3d = buildings_joined.copy()

# Function to add Z-height coordinate to polygon vertices
def make_3d_polygon(row):
    geom = row.geometry
    height = row['final_height_m']
    if geom is None or geom.is_empty:
        return geom
    # Assign the height value to the Z coordinate of every vertex
    return transform(lambda x, y, z=None: (x, y, height), geom)

# Apply 3D geometry transformation
buildings_3d['geometry'] = buildings_3d.apply(make_3d_polygon, axis=1)

# Convert back to WGS84 for GeoJSON export
polygons_3d_wgs84 = buildings_3d.to_crs("EPSG:4326")

# Drop unnecessary columns
polygons_3d_wgs84 = polygons_3d_wgs84.drop(
    columns=[
        "index_right", "index_left", "ext_height", "canton", "commune_nom",
        "femmes_mineures", "hommes_mineurs", "femmes_majeures", "hommes_majeurs",
        "population","total_commune_volume_m3", "vol_weight"
    ],
    errors="ignore"
)

# Export to 3D GeoJSON
polygons_3d_wgs84.to_file("buildings_3d_polygons_eddie.geojson", driver="GeoJSON")

print("Successfully exported 3D roof polygon geometries!")

Successfully exported 3D roof polygon geometries!
